<a href="https://colab.research.google.com/github/AnanyaAsthana/Hadoop-CUDA-Lab/blob/main/cudalab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvcc --version


nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [ ]:
%%writefile vector_add.cu
#include <stdio.h>
#include <cuda.h>
#include <stdlib.h>

__global__ void vectorAdd(float *A, float *B, float *C, int n)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    if (tid < n)
        C[tid] = A[tid] + B[tid];
}

int main()
{
    int n;
    printf("Enter number of elements: ");
    scanf("%d", &n);

    size_t size = n * sizeof(float);

    float *h_A = (float*)malloc(size);
    float *h_B = (float*)malloc(size);
    float *h_C = (float*)malloc(size);

    for(int i = 0; i < n; i++) {
        h_A[i] = 1.0f;
        h_B[i] = 2.0f;
    }

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, size);
    cudaMalloc(&d_B, size);
    cudaMalloc(&d_C, size);

    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    printf("\nThreadsPerBlock\tBlocks\tGPU Time (ms)\n");
    printf("-------------------------------------------------\n");

    // Test different thread configurations
    for(int threads = 32; threads <= 1024; threads *= 2)
    {
        int blocks = (n + threads - 1) / threads;

        cudaEvent_t start, stop;
        cudaEventCreate(&start);
        cudaEventCreate(&stop);

        cudaEventRecord(start);

        vectorAdd<<<blocks, threads>>>(d_A, d_B, d_C, n);

        cudaEventRecord(stop);
        cudaEventSynchronize(stop);

        float time;
        cudaEventElapsedTime(&time, start, stop);

        printf("%d\t\t%d\t%f\n", threads, blocks, time);

        cudaEventDestroy(start);
        cudaEventDestroy(stop);
    }

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    free(h_A);
    free(h_B);
    free(h_C);

    return 0;
}



Overwriting vector_add.cu


In [ ]:
!nvcc vector_add.cu -o vector_add



nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./vector_add


Enter number of elements: 500

ThreadsPerBlock	Blocks	GPU Time (ms)
-------------------------------------------------
32		16	18.417088
64		8	0.010016
128		4	0.008160
256		2	0.008000
512		1	0.007936
1024		1	0.007776


In [ ]:
%%writefile vector_reduce_perf.cu

#include <stdio.h>
#include <cuda.h>
#include <stdlib.h>
#include <limits.h>

__global__ void reduceKernel(int *input, int *sum, int n)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    if (tid < n)
        atomicAdd(sum, input[tid]);
}

int main()
{
    int n;
    printf("Enter number of elements: ");
    scanf("%d", &n);

    size_t size = n * sizeof(int);

    int *h_input = (int*)malloc(size);
    for(int i = 0; i < n; i++)
        h_input[i] = 1;

    int *d_input, *d_sum;
    cudaMalloc(&d_input, size);
    cudaMalloc(&d_sum, sizeof(int));

    cudaMemcpy(d_input, h_input, size, cudaMemcpyHostToDevice);

    printf("\nThreadsPerBlock\tBlocks\tGPU Time (ms)\n");
    printf("-------------------------------------------------\n");

    for(int threads = 32; threads <= 1024; threads *= 2)
    {
        int blocks = (n + threads - 1) / threads;

        int zero = 0;
        cudaMemcpy(d_sum, &zero, sizeof(int), cudaMemcpyHostToDevice);

        cudaEvent_t start, stop;
        cudaEventCreate(&start);
        cudaEventCreate(&stop);

        cudaEventRecord(start);

        reduceKernel<<<blocks, threads>>>(d_input, d_sum, n);

        cudaEventRecord(stop);
        cudaEventSynchronize(stop);

        float time;
        cudaEventElapsedTime(&time, start, stop);

        printf("%d\t\t%d\t%f\n", threads, blocks, time);

        cudaEventDestroy(start);
        cudaEventDestroy(stop);
    }

    cudaFree(d_input);
    cudaFree(d_sum);
    free(h_input);

    return 0;
}


Writing vector_reduce_perf.cu


In [ ]:
!nvcc vector_add_perf.cu -o add
!./add


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
cc1plus: fatal error: vector_add_perf.cu: No such file or directory
compilation terminated.
/bin/bash: line 1: ./add: No such file or directory


In [ ]:
!nvcc vector_reduce_perf.cu -o reduce
!./reduce



nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Enter number of elements: 500

ThreadsPerBlock	Blocks	GPU Time (ms)
-------------------------------------------------
32		16	13.957056
64		8	0.011040
128		4	0.009088
256		2	0.008288
512		1	0.008160
1024		1	0.008192
